In [1]:
import pandas as pd
import numpy as np

import datetime
import os, sys
import importlib
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import utils
importlib.reload(utils)


from utils import plot_series, plot_series_with_names, plot_series_bar
from utils import plot_dataframe
from utils import get_universe_adjusted_series, scale_weights_to_one, scale_to_book_long_short
from utils import generate_portfolio, backtest_portfolio
from utils import match_implementations

import plotly.graph_objects as go

In [2]:
# This directory can be used if you're working on a Kaggle Notebook inside the competition
# Change the directory as per your requirements if you're working somewhere else
data_dir = r"/kaggle/input/competitions/qrt-quant-challenge-2026-iit-delhi"

features = pd.read_parquet(os.path.join(data_dir, "features.parquet"))

universe = pd.read_parquet(os.path.join(data_dir, "universe.parquet"))
 
returns = pd.read_parquet(os.path.join(data_dir, "returns.parquet"))

In [3]:
import pandas as pd
import numpy as np
import xgboost as xgb
# Note: Ensure you have already imported match_implementations from utils

# ==========================================
# 1. WATER-FILL NORMALIZATION (HELPER)
# ==========================================
def cap_and_redistribute(w_raw, max_w=0.10, target=0.5):
    """Water-fill algorithm to cap weights at 10% and redistribute excess"""
    w = w_raw.copy()
    
    # Initial Normalization
    row_sums = w.sum(axis=1).replace(0, np.nan)
    w = w.div(row_sums, axis=0).fillna(0) * target
    
    # Redistribute loop
    for _ in range(5):
        w_capped = w.clip(upper=max_w)
        shortfall = target - w_capped.sum(axis=1)
        uncapped_mask = (w_capped < max_w) & (w_capped > 0)
        uncapped_sum = w_capped.where(uncapped_mask, 0).sum(axis=1).replace(0, np.nan)
        scale_factor = 1.0 + (shortfall / uncapped_sum)
        w_scaled = w_capped.multiply(scale_factor, axis=0)
        w = w_capped.where(~uncapped_mask, w_scaled)
        
    return w.fillna(0)


# ==========================================
# 2. MASTER TRAINING (EXECUTE THIS FIRST)
# ==========================================
print("--- Training Master Fast Model ---")

# Use ALL raw features (The proven 1.732 ensemble diversity setup)
fast_features_list = list(features.columns.levels[0])

# Using 2012-2019 to match your best Walk-Forward configuration
TRAIN_START = "2005-01-01"
TRAIN_END = "2019-12-31"

target_forward = returns.shift(-1)

X_list_train = []
for f in fast_features_list:
    feat_df = features[f].shift(1).loc[TRAIN_START:TRAIN_END].where(universe == 1).rank(axis=1, pct=True)
    X_list_train.append(feat_df.stack(dropna=False).rename(f))
    
X_train = pd.concat(X_list_train, axis=1)
y_train = target_forward.loc[TRAIN_START:TRAIN_END].stack().rename("target_return")

train_data = X_train.join(y_train, how="inner").dropna()

# Locked XGBoost Hyperparameters
master_xgb = xgb.XGBRegressor(
    n_estimators=100,         # Start small, increase if early stopping allows
    learning_rate=0.05,       # Slow learning rate
    max_depth=3,              # SHALLOW trees to prevent overfitting
    subsample=0.2,            # Randomly sample 20% of rows per tree
    colsample_bytree=0.5,     # Randomly sample 50% of features per tree
    objective='reg:squarederror', 
    n_jobs=-1,
    random_state=42
)

master_xgb.fit(train_data[fast_features_list], train_data["target_return"])
print("Master Model Training Complete.\n")


# ==========================================
# 3. VECTORIZED GENERATOR (FOR SUBMISSION)
# ==========================================
# Note the default arguments for seamless integration with QRT utils
def generate_portfolio_vectorized(features, universe, model=master_xgb, feature_names=fast_features_list, start_date=None, end_date=None):
    if start_date is None: start_date = returns.index[0]
    if end_date is None: end_date = returns.index[-1]
        
    print(f"Generating Vectorized Portfolio ({start_date} to {end_date})...")
    universe_mask = universe.loc[start_date:end_date].astype(bool)
    
    X_list = []
    for f in feature_names:
        feat_df = features[f].shift(1).loc[start_date:end_date].where(universe_mask).rank(axis=1, pct=True)
        X_list.append(feat_df.stack(dropna=False).rename(f))
        
    X_test = pd.concat(X_list, axis=1)
    valid_rows = X_test.notna().all(axis=1)
    
    preds_valid = model.predict(X_test[valid_rows][feature_names])
    predictions_1d = pd.Series(np.nan, index=X_test.index)
    predictions_1d.loc[valid_rows] = preds_valid
    predictions = predictions_1d.unstack(level=1)
    
    ranks = predictions.rank(axis=1, pct=True)
    
    # Asymmetric Execution (Top 2.5% Long, Bottom 5% Short)
    long_mask = ranks >= 0.975
    short_mask = ranks <= 0.050
    
    long_preds = predictions.where(long_mask, 0)
    short_preds = predictions.where(short_mask, 0).abs()
    
    # Signal Dispersion Scaling
    daily_max_long = long_preds.max(axis=1) + 1e-6
    daily_max_short = short_preds.max(axis=1) + 1e-6
    
    long_confidence = long_preds.div(daily_max_long, axis=0).where(universe_mask, 0).abs()
    short_confidence = short_preds.div(daily_max_short, axis=0).where(universe_mask, 0).abs()
    
    # Apply Cap and Redistribute
    final_longs = cap_and_redistribute(long_confidence, max_w=0.10, target=0.5)
    final_shorts = cap_and_redistribute(short_confidence, max_w=0.10, target=0.5) * -1.0
    
    return final_longs + final_shorts


# ==========================================
# 4. ITERATIVE GENERATOR (FOR COMPLIANCE)
# ==========================================
def get_weights(features_step, universe_step, model=master_xgb, feature_names=fast_features_list):
    """Iterative daily function for match_implementations"""
    if isinstance(features_step, pd.DataFrame):
        features_step = features_step.iloc[-1]
    if isinstance(universe_step, pd.DataFrame):
        universe_step = universe_step.iloc[-1]
        
    universe_mask = universe_step.astype(bool)
    X_today = pd.DataFrame(index=universe_mask.index)
    
    for f in feature_names:
        feat_raw = features_step[f].where(universe_mask)
        X_today[f] = feat_raw.rank(pct=True)
        
    valid_rows = X_today.notna().all(axis=1)
    if not valid_rows.any():
        return pd.Series(0, index=universe_mask.index)

    preds_valid = model.predict(X_today[valid_rows][feature_names])
    predictions = pd.Series(np.nan, index=universe_mask.index)
    predictions.loc[valid_rows] = preds_valid
    
    ranks = predictions.rank(pct=True)
    
    long_mask = ranks >= 0.975
    short_mask = ranks <= 0.050
    
    long_preds = predictions.where(long_mask, 0)
    short_preds = predictions.where(short_mask, 0).abs()
    
    daily_max_long = long_preds.max() + 1e-6
    daily_max_short = short_preds.max() + 1e-6
    
    long_confidence = (long_preds / daily_max_long).where(universe_mask, 0).abs()
    short_confidence = (short_preds / daily_max_short).where(universe_mask, 0).abs()
    
    long_df = pd.DataFrame([long_confidence])
    short_df = pd.DataFrame([short_confidence])
    
    final_longs_df = cap_and_redistribute(long_df, max_w=0.10, target=0.5)
    final_shorts_df = cap_and_redistribute(short_df, max_w=0.10, target=0.5) * -1.0
    
    # Combine longs and shorts
    final_weights = final_longs_df.iloc[0] + final_shorts_df.iloc[0]
    final_weights = final_weights.fillna(0)
    
    valid_stocks_mask = universe_step == 1
    compliant_dict = final_weights[valid_stocks_mask].to_dict()
    
    return compliant_dict


# ==========================================
# 5. EXECUTION & CSV GENERATION
# ==========================================
# Generate full 20-year portfolio
FULL_START = universe.index[0]
FULL_END = universe.index[-1]

submission_portfolio = generate_portfolio_vectorized(
    features, 
    universe, 
    start_date=FULL_START,
    end_date=FULL_END
)

# Round to 6 decimals to save file size
submission_portfolio = submission_portfolio.round(6)

# Save to CSV
submission_filename = "submission.csv"
submission_portfolio.to_csv(submission_filename)

print(f"\nSuccess! Full 20-year portfolio weights saved to {submission_filename}")
print(f"Total days generated: {len(submission_portfolio)}")
print(f"Sanity Check - Gross Exposure Day 100: {submission_portfolio.iloc[100].abs().sum():.4f} (Should be exactly 1.0)")

# ==========================================
# 6. RUN QRT COMPLIANCE CHECK
# ==========================================
# This proves your fast code and iterative code do the exact same thing
print("\nRunning Compliance Check (match_implementations)...")
match_implementations(get_weights, submission_portfolio, features, universe, returns)

--- Training Master Fast Model ---
Master Model Training Complete.

Generating Vectorized Portfolio (2005-01-03 00:00:00 to 2025-02-07 00:00:00)...

Success! Full 20-year portfolio weights saved to submission.csv
Total days generated: 5058
Sanity Check - Gross Exposure Day 100: 1.0000 (Should be exactly 1.0)

Running Compliance Check (match_implementations)...
Starting to generate Iterative Portfolio


100%|██████████| 41/41 [00:11<00:00,  3.63it/s]


Iterative Portfolio Generated
Correlation of 0.9999999999883136 between Iterative and Vectorized Implementations. Both implementations match!
